**정해진 기간 내의 휴장일 설정 및 휴장일 셀 설정**

In [ ]:
# STEP 1: 라이브러리 로드
import pandas as pd
from google.colab import files
from io import StringIO
from datetime import datetime

# STEP 2: 휴장일 리스트 (2022~2024년 기준)
holiday_set = set([
    # 2022
    '2022-01-01', '2022-02-01', '2022-02-02', '2022-02-03',
    '2022-03-01', '2022-05-05', '2022-05-08', '2022-06-06',
    '2022-08-15', '2022-09-10', '2022-09-11', '2022-09-12',
    '2022-10-03', '2022-10-09', '2022-12-25', '2022-12-31',
    # 2023
    '2023-01-01', '2023-01-22', '2023-01-23', '2023-01-24',
    '2023-03-01', '2023-05-05', '2023-05-06', '2023-05-27',
    '2023-06-06', '2023-08-15', '2023-09-28', '2023-09-29', '2023-09-30',
    '2023-10-03', '2023-10-09', '2023-12-25', '2023-12-31',
    # 2024
    '2024-01-01', '2024-02-10', '2024-02-11', '2024-02-12',
    '2024-03-01', '2024-05-01', '2024-05-05', '2024-05-06', '2024-05-15',
    '2024-06-06', '2024-08-15', '2024-09-16', '2024-09-17', '2024-09-18',
    '2024-10-03', '2024-10-09', '2024-12-25', '2024-12-31'
])

# STEP 3: 파일 업로드
print("📁 CSV 파일을 업로드하세요")
uploaded = files.upload()

# STEP 4: 파일 처리
for filename in uploaded.keys():
    # CSV 읽기
    df = pd.read_csv(StringIO(uploaded[filename].decode('utf-8')))

    # 날짜 자동 파싱
    df['Date_Parsed'] = pd.to_datetime(df.iloc[:, 0], errors='coerce')

    # 휴장일 여부 계산
    market_closed = df['Date_Parsed'].apply(
        lambda x: 1 if pd.isnull(x) else (
            1 if (x.strftime('%Y-%m-%d') in holiday_set or x.weekday() >= 5) else 0
        )
    )

    # D열 (4번째 열) 덮어쓰기
    df.iloc[:, 3] = market_closed

    # 중간 처리용 컬럼 제거
    df.drop(columns=['Date_Parsed'], inplace=True)

    # 새 파일명으로 저장 및 다운로드
    new_filename = filename.replace('.csv', '_holiday.csv')
    df.to_csv(new_filename, index=False)
    files.download(new_filename)


📁 CSV 파일을 업로드하세요


Saving Hyundai_score_sum_holiday.csv to Hyundai_score_sum_holiday (1).csv
Saving Kbfg_score_sum_holiday.csv to Kbfg_score_sum_holiday (1).csv
Saving Kia_score_sum_holiday.csv to Kia_score_sum_holiday (1).csv
Saving Lgchem_score_sum_holiday.csv to Lgchem_score_sum_holiday (1).csv
Saving Lgensol_score_sum_holiday.csv to Lgensol_score_sum_holiday (1).csv
Saving Naver_score_sum_holiday.csv to Naver_score_sum_holiday (1).csv
Saving PoscoHoldings_score_sum_holiday.csv to PoscoHoldings_score_sum_holiday (1).csv
Saving Samsung_score_sum_holiday.csv to Samsung_score_sum_holiday (1).csv
Saving SamsungBiologics_score_sum_holiday.csv to SamsungBiologics_score_sum_holiday (1).csv
Saving Skhynix_score_sum_holiday.csv to Skhynix_score_sum_holiday (1).csv


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

...

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

**휴장일 기준으로 화제성 점수 합치기 및 점수 정규화(Min-Max)**

In [ ]:
from google.colab import files
import pandas as pd
import io
import os

# 파일 업로드
uploaded = files.upload()

for filename in uploaded.keys():
    df = pd.read_csv(io.BytesIO(uploaded[filename]))

    # 정확한 열 이름으로 할당 (4개 열)
    df.columns = ['A', 'B', 'E', 'D']  # 날짜, 점수, 요일, 공휴일여부

    # 결과 저장용 리스트
    result_rows = []
    i = 0
    while i < len(df):
        if df.iloc[i]['D'] == 0:
            group_start = i
            acc_b = df.iloc[i]['B']
            days = [df.iloc[i]['E']]
            count = 1
            i += 1
            while i < len(df) and df.iloc[i]['D'] == 1:
                acc_b += df.iloc[i]['B']
                days.append(df.iloc[i]['E'])
                count += 1
                i += 1
            avg_b = acc_b / count
            combined_day = ''.join(days)
            result_rows.append([
                df.iloc[group_start]['A'],  # 날짜
                avg_b,                      # 평균 점수
                0,                          # 의미없는 C열 자리
                0,                          # D: 그룹의 대표이므로 0
                '',                         # E: 나중에 정규화 점수
                combined_day               # F: 합쳐진 요일
            ])
        else:
            i += 1

    # 결과 DataFrame
    result_df = pd.DataFrame(result_rows, columns=['A', 'B', 'C', 'D', 'E', 'F'])

    # B 정규화
    b_min = result_df['B'].min()
    b_max = result_df['B'].max()
    result_df['E'] = (result_df['B'] - b_min) / (b_max - b_min)

    # 열 순서 정리: 날짜, 평균점수, C, 정규화(E), 요일(F), 공휴일여부(D)
    result_df = result_df[['A', 'B', 'C', 'E', 'F', 'D']]

    # 저장 및 다운로드
    base_name = os.path.splitext(filename)[0]
    new_filename = base_name + '_sum.csv'
    result_df.to_csv(new_filename, index=False)
    files.download(new_filename)


Saving Lgensol_score_sum_holiday (1)_holiday.csv to Lgensol_score_sum_holiday (1)_holiday (2).csv
Saving Naver_score_sum_holiday (1)_holiday.csv to Naver_score_sum_holiday (1)_holiday (2).csv
Saving PoscoHoldings_score_sum_holiday (1)_holiday.csv to PoscoHoldings_score_sum_holiday (1)_holiday (2).csv
Saving Samsung_score_sum_holiday (1)_holiday.csv to Samsung_score_sum_holiday (1)_holiday (2).csv
Saving SamsungBiologics_score_sum_holiday (1)_holiday.csv to SamsungBiologics_score_sum_holiday (1)_holiday (2).csv
Saving Skhynix_score_sum_holiday (1)_holiday.csv to Skhynix_score_sum_holiday (1)_holiday (2).csv
Saving Hyundai_score_sum_holiday (1)_holiday.csv to Hyundai_score_sum_holiday (1)_holiday (2).csv
Saving Kbfg_score_sum_holiday (1)_holiday.csv to Kbfg_score_sum_holiday (1)_holiday (2).csv
Saving Kia_score_sum_holiday (1)_holiday.csv to Kia_score_sum_holiday (1)_holiday (2).csv
Saving Lgchem_score_sum_holiday (1)_holiday.csv to Lgchem_score_sum_holiday (1)_holiday (2).csv


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

...

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

**감성 분석 CSV 파일에 휴장일 추가**

In [ ]:
# STEP 1: 라이브러리 로드
import pandas as pd
from google.colab import files
from io import StringIO
from datetime import datetime

# STEP 2: 휴장일 리스트 (2022~2024년 기준)
holiday_set = set([
    # 2022
    '2022-01-01', '2022-02-01', '2022-02-02', '2022-02-03',
    '2022-03-01', '2022-05-05', '2022-05-08', '2022-06-06',
    '2022-08-15', '2022-09-10', '2022-09-11', '2022-09-12',
    '2022-10-03', '2022-10-09', '2022-12-25', '2022-12-31',
    # 2023
    '2023-01-01', '2023-01-22', '2023-01-23', '2023-01-24',
    '2023-03-01', '2023-05-05', '2023-05-06', '2023-05-27',
    '2023-06-06', '2023-08-15', '2023-09-28', '2023-09-29', '2023-09-30',
    '2023-10-03', '2023-10-09', '2023-12-25', '2023-12-31',
    # 2024
    '2024-01-01', '2024-02-10', '2024-02-11', '2024-02-12',
    '2024-03-01', '2024-05-01', '2024-05-05', '2024-05-06', '2024-05-15',
    '2024-06-06', '2024-08-15', '2024-09-16', '2024-09-17', '2024-09-18',
    '2024-10-03', '2024-10-09', '2024-12-25', '2024-12-31'
])

# STEP 3: 파일 업로드
print("📁 CSV 파일을 업로드하세요")
uploaded = files.upload()

# STEP 4: 파일 처리
for filename in uploaded.keys():
    # CSV 불러오기
    df = pd.read_csv(StringIO(uploaded[filename].decode('utf-8')))

    # ✅ 원본 미리보기 출력
    print(f"\n📄 원본 데이터 미리보기: {filename}")
    display(df.head())

    # 날짜 컬럼 처리 (첫 번째 열이 YYYYMMDD 형식이라고 가정)
    df['Date_Parsed'] = pd.to_datetime(df.iloc[:, 0].astype(str), format='%Y%m%d', errors='coerce')

    # 요일 추가
    df['Weekday'] = df['Date_Parsed'].dt.day_name()

    # 휴장 여부 추가
    df['Market_Closed'] = df['Date_Parsed'].apply(
        lambda x: 1 if pd.isnull(x) else (
            1 if (x.strftime('%Y-%m-%d') in holiday_set or x.weekday() >= 5) else 0
        )
    )

    # 중간 컬럼 제거
    df.drop(columns=['Date_Parsed'], inplace=True)

    # ✅ 처리 후 미리보기
    print(f"\n✅ 처리된 데이터 미리보기: {filename}")
    display(df.head())

    # 새 파일명 저장 및 다운로드
    new_filename = filename.replace('.csv', '_holiday.csv')
    df.to_csv(new_filename, index=False)
    files.download(new_filename)


**감성 분석 CSV파일 휴장일 기준으로 합치기**

In [ ]:
from google.colab import files
import pandas as pd
import io
import os
import zipfile

# 업로드
uploaded = files.upload()

processed_files = []

for filename in uploaded.keys():
    df = pd.read_csv(io.BytesIO(uploaded[filename]))
    df.columns = ['A', 'B', 'C', 'D', 'E', 'F']

    result_rows = []
    i = 0
    while i < len(df):
        row = df.iloc[i]
        if row['F'] == 0:
            group_start = i
            acc_b = row['B']
            acc_c = row['C']
            days = [row['E']]
            count = 1
            i += 1
            while i < len(df) and df.iloc[i]['F'] != 0:
                acc_b += df.iloc[i]['B']
                acc_c += df.iloc[i]['C']
                days.append(df.iloc[i]['E'])
                count += 1
                i += 1
            avg_b = acc_b / count
            avg_c = acc_c / count
            combined_day = ''.join(days)
            result_rows.append([df.iloc[group_start]['A'], avg_b, avg_c, combined_day, 0])
        else:
            i += 1

    result_df = pd.DataFrame(result_rows, columns=['A', 'B', 'C', 'E', 'F'])

    base, ext = os.path.splitext(filename)
    new_filename = f"{base}_화제성_sum.csv"
    result_df.to_csv(new_filename, index=False)
    processed_files.append(new_filename)

# 압축 후 다운로드
zip_filename = "processed_files.zip"
with zipfile.ZipFile(zip_filename, 'w') as zipf:
    for file in processed_files:
        zipf.write(file)

files.download(zip_filename)


**1군 뉴스사에 대한 3의 가중치 적용**

In [ ]:
from google.colab import files
import pandas as pd
import io
import os
import zipfile

# 파일 업로드
uploaded = files.upload()

# 변환된 파일 저장 목록
processed_files = []

for filename in uploaded.keys():
    # CSV 읽기 (1행을 그대로 헤더로 사용)
    df = pd.read_csv(io.BytesIO(uploaded[filename]))

    # B열과 C열만 3배 처리 (열 이름 또는 위치로 접근 가능)
    df.iloc[:, 1] = df.iloc[:, 1] * 3  # B열
    df.iloc[:, 2] = df.iloc[:, 2] * 3  # C열

    # 저장 파일명 지정
    base, ext = os.path.splitext(filename)
    new_filename = f"{base}_triple.csv"
    df.to_csv(new_filename, index=False)
    processed_files.append(new_filename)

# ZIP으로 묶기
zip_filename = "tripled_files.zip"
with zipfile.ZipFile(zip_filename, 'w') as zipf:
    for file in processed_files:
        zipf.write(file)

# 다운로드
files.download(zip_filename)


**1군 뉴스사의 점수와 2군 뉴스사의 점수 합치기**

In [ ]:
from google.colab import files
import pandas as pd
import io
import os
import os.path

# 파일 업로드
uploaded = files.upload()
file_names = list(uploaded.keys())

# 파일 2개 불러오기
df1 = pd.read_csv(io.BytesIO(uploaded[file_names[0]]))
df2 = pd.read_csv(io.BytesIO(uploaded[file_names[1]]))

# outer join으로 날짜 기준 병합
merged = pd.merge(df1, df2, on='A', how='outer', suffixes=('_1', '_2'))

# B, C는 결측값 0으로 채우고 더함
merged['B'] = merged['B_1'].fillna(0) + merged['B_2'].fillna(0)
merged['C'] = merged['C_1'].fillna(0) + merged['C_2'].fillna(0)

# E, F는 첫 번째 파일 기준 사용
merged['E'] = merged['E_1']
merged['F'] = merged['F_1']

# 결과 열 정리
final_df = merged[['A', 'B', 'C', 'E', 'F']]

# 공통 접두사 추출
def longest_common_prefix(s1, s2):
    from os.path import commonprefix
    return os.path.commonprefix([s1, s2])

prefix = longest_common_prefix(file_names[0], file_names[1]).rstrip('_').rstrip('.csv')
output_filename = f"{prefix}_merged_result.csv"

# 저장 및 다운로드
final_df.to_csv(output_filename, index=False)
files.download(output_filename)


**합친 점수를 4로 나누어 평균 처리**

In [ ]:
from google.colab import files
import pandas as pd
import io
import os
import zipfile

# 파일 업로드
uploaded = files.upload()

# 처리된 파일들 저장
processed_files = []

for filename in uploaded.keys():
    df = pd.read_csv(io.BytesIO(uploaded[filename]))

    # B, C 열을 4로 나누기
    df.iloc[:, 1] = df.iloc[:, 1] / 4  # B열
    df.iloc[:, 2] = df.iloc[:, 2] / 4  # C열

    # 새 파일 이름 저장
    base, ext = os.path.splitext(filename)
    new_filename = f"{base}_AVG.csv"
    df.to_csv(new_filename, index=False)
    processed_files.append(new_filename)

# zip으로 묶기
zip_filename = "avg_files.zip"
with zipfile.ZipFile(zip_filename, 'w') as zipf:
    for file in processed_files:
        zipf.write(file)

# 다운로드
files.download(zip_filename)
